# SPACEc: Cell Segmentation

The aim of cell segmentation is to identify each cell within a given image and derive single-cell data from the image. In the current version of SPACEc, segmentation is performed using **Cellpose**, a deep learning-enabled segmentation algorithm that provides different pre-trained models which can be directly employed. Additionally, Cellpose allows users to train their own models for custom segmentation tasks. SPACEc supports both classic Cellpose (cyto3) and the newer Cellpose-SAM (cpsam) generalist model — see the "Choosing your Cellpose backend" section below. Previous versions of SPACEc also supported **Deepcell Mesmer**, but Mesmer compatibility has been dropped in the current version.

For the purpose of this tutorial, we will use **Cellpose**.

The steps of the script are:

1. Deciding on the image channels for segmentation  
2. Running segmentation with Cellpose  
3. Quality control of the segmented images  
4. Storing the data for further processing

In [ ]:
# import spacec first
import spacec as sp

#import standard packages
import os
import warnings
import matplotlib
import pickle
warnings.filterwarnings('ignore')


In [ ]:
# Apply SPACEc publication style to all plots in this notebook
sp.pl.set_style(sp.pl.SPACEcStyle.presentation())

In [ ]:
# Specify the path to the data
data_path = "path/to/example_data/raw/"
output_dir = "path/to/output/"
os.makedirs(output_dir, exist_ok=True)

If you want to use GPU acceleration for the segmentation you should ensure that CUDA is installed and the GPU detected by the python environment. If you have a compatible Nvidia card but never installed CUDA Toolkit before you can find installation instructions here: https://developer.nvidia.com/cuda-downloads

In [ ]:
#check if GPU availability (only applicable if you have an Nvidia GPU installed)
!nvcc --version
!nvidia-smi

In [ ]:
# Check if GPU is available to PyTorch and SPACEc. If not, SPACEc will default to CPU processing.
sp.hf.check_for_gpu()

## Cell segmentation

### Choosing your Cellpose backend

SPACEc supports two generations of Cellpose for segmentation:

- **cyto3** — the classic per-tissue Cellpose model zoo (cellpose 3.x)
- **Cellpose-SAM** (`cpsam`) — a single SAM-based generalist model (cellpose>=4.x), largely diameter- and channel-invariant

Which one you get is determined by **which cellpose package is installed** in your environment, not by a SPACEc flag. `model=None` (the default below) auto-selects the right generalist model for whichever cellpose is installed.

| You install | `model=None` gives you | Legacy names (`cyto3`, `nuclei`) | v3 fine-tuned custom models |
| --- | --- | --- | --- |
| `cellpose>=3,<4` | `cyto3` | work natively | work |
| `cellpose>=4` | `cpsam` | remapped to `cpsam` (with a warning) | **not loadable** |

See [docs/cellpose_models.md](../../docs/cellpose_models.md) (or the "Cellpose Models" page in the SPACEc docs) for the full guide, including when to prefer one over the other and what changes in your `cell_segmentation` call.

**Custom models:** if you plan to load a custom fine-tuned model further below, note it must have been trained/exported with the *same* cellpose generation you have installed — a cellpose-3 `.zip` checkpoint cannot be loaded by Cellpose-SAM.


In [ ]:
# Check what's currently installed, and preview what switching would involve — this makes NO changes.
sp.hf.set_cellpose_version(4, dry_run=True)


In [ ]:
# To actually switch cellpose versions, uncomment and run the line below, then RESTART YOUR KERNEL.
# Switching does not take effect in this running process (cellpose/torch are already imported),
# and set_cellpose_version() will explain the consequences and ask for confirmation before installing.
# sp.hf.set_cellpose_version(4)  # or 3 / "cyto3" / "sam" / "cpsam"


**NOTE:** Our segmentation function features a parameter called 'input_format'. This parameter defines what input data the function accepts. If set to 'Multichannel' the function expects a single multichannel tiff file, if set to 'Channels' the function expects a folder with single Tiff files (no channelnames.txt required) and if set to 'CODEX' the function reads the output of the classic first gen CODEX setup.

Before committing to potentially time intense segmentation it might be useful to visualize the segmentation channels. In this tutorial we provide both nuclei and membrane channels. Especially if no general membrane marker is available it is useful to combine membrane markers as shown below. 

In [ ]:
# (optional, one can just use nuclei for segmentation)
# Visualize membrane channels to use for cell segmentation 

sp.pl.segmentation_ch(
    file_name = output_dir + 'reg001_X01_Y01_Z01.tif', # image for segmentation
    channel_file = data_path + 'channelnames.txt', # all channels used for staining
    output_dir = output_dir, #
    membrane_channel_list = ["CD45", "betaCatenin"], #default is None; if provide more than one channel, then they will be combined
    nuclei_channel = 'DAPI', # channel to use for nuclei segmentation
    input_format = 'Multichannel', 
    um_per_px = 0.5, # pixel size in microns, used for scaling the image for segmentation
)

After deciding on the channels for segmentation, segmentation is performed with the 'cell_segmentation' function. Besides choosing the channels the function allows to select the segmentation model. The function expects a multichannel tif file and a channel names file as input (please see our example data as an example). The segmentation output is stored as csv file.

This tutorial uses two regions (`reg001` -> "tonsil1", `reg002` -> "tonsil2") throughout notebooks 03.1/03.2/04, so we segment both here, one `cell_segmentation` call per region. `save_mask=True` additionally writes each region's raw label mask to `{output_fname}_mask.tif`, which lets later steps (e.g. spillover compensation in 03.2) load a region's mask by path via `sp.hf.hf_load_masks` instead of unpickling this whole `seg_output` dict.

In [ ]:
seg_output1 = sp.tl.cell_segmentation(
    file_name = output_dir + 'reg001_X01_Y01_Z01.tif',
    channel_file = data_path + 'channelnames.txt',
    output_dir = output_dir,
    output_fname = 'tonsil1',
    seg_method ='cellpose', # cellpose or mesmer
    model=None, # None = auto-select ('cpsam' on cellpose>=4, 'cyto3' on cellpose 3.x) -- see 'Choosing your Cellpose backend' above
    diameter=28, # average cell diameter (in pixels). If set to None, it will be automatically estimated. Cellpose-SAM still uses this to rescale toward its training mean.
    flow_threshold=0.4, # Cellpose-SAM tuning knob; higher = more (less strict) masks
    cellprob_threshold=0.0, # Cellpose-SAM tuning knob; lower = more/larger masks
    nuclei_channel = 'DAPI',
    membrane_channel_list = ["CD45", "betaCatenin"], #default is None #default is None; if provide more than one channel, then they will be combined
    input_format ='Multichannel', # Phenocycler or codex
    resize_factor=1, # default is 1; if the image is too large, lower the value. Lower values will speed up the segmentation but may reduce the accuracy.
    size_cutoff = 0,
    save_mask = True) # writes tonsil1_mask.tif to output_dir -- lets 03.1/03.2 load this region's mask by path (sp.hf.hf_load_masks) instead of unpickling this whole seg_output dict
# Note: Cellpose-SAM downloads its weights (~GB) on first use and needs more GPU/VRAM than cyto3;
# if you hit an out-of-memory error, lower tile_size (e.g. to 1024-2048) and/or batch_size.


In [ ]:
seg_output2 = sp.tl.cell_segmentation(
    file_name = output_dir + 'reg002_X01_Y01_Z01.tif',
    channel_file = data_path + 'channelnames.txt',
    output_dir = output_dir,
    output_fname = 'tonsil2',
    seg_method ='cellpose', # cellpose or mesmer
    model=None, # None = auto-select ('cpsam' on cellpose>=4, 'cyto3' on cellpose 3.x) -- see 'Choosing your Cellpose backend' above
    diameter=28, # average cell diameter (in pixels). If set to None, it will be automatically estimated. Cellpose-SAM still uses this to rescale toward its training mean.
    flow_threshold=0.4, # Cellpose-SAM tuning knob; higher = more (less strict) masks
    cellprob_threshold=0.0, # Cellpose-SAM tuning knob; lower = more/larger masks
    nuclei_channel = 'DAPI',
    membrane_channel_list = ["CD45", "betaCatenin"], #default is None #default is None; if provide more than one channel, then they will be combined
    input_format ='Multichannel', # Phenocycler or codex
    resize_factor=1, # default is 1; if the image is too large, lower the value. Lower values will speed up the segmentation but may reduce the accuracy.
    size_cutoff = 0,
    save_mask = True) # writes tonsil2_mask.tif to output_dir
# Note: Cellpose-SAM downloads its weights (~GB) on first use and needs more GPU/VRAM than cyto3;
# if you hit an out-of-memory error, lower tile_size (e.g. to 1024-2048) and/or batch_size.


You can also load a custom fine-tuned cellpose model into SPACEc as shown below. The required input file is the zip file that cellpose outputs after training.

**Note:** a custom model must be trained with the same Cellpose generation you have installed. A model fine-tuned on cellpose 3.x (`cyto3`-based) cannot be loaded by Cellpose-SAM (cellpose>=4) — it uses a different backbone. Check `sp.hf.set_cellpose_version(3, dry_run=True)` if you need to confirm/switch to cellpose 3.x for this.

In [ ]:
# This codeblock only runs if you have a custom model to use for segmentation. If you don't have a custom model, you can skip this codeblock and use the default Cellpose model instead.

# check if custom model file exists
custom_model_path = '/home/user/path_to_custom_model/models/CP_XXXX_XXXX'
if not os.path.exists(custom_model_path):
    raise FileNotFoundError(f"Custom model file not found at {custom_model_path}.")

seg_output_cellpose = sp.tl.cell_segmentation(
    file_name = output_dir + 'reg002_X01_Y01_Z01.tif',
    channel_file = data_path + 'channelnames.txt',
    output_dir = output_dir,
    output_fname = 'tonsil2',
    seg_method ='cellpose', # cellpose or mesmer
    model=custom_model_path, # cellpose model
    diameter=28, # average cell diameter (in pixels). If set to None, it will be automatically estimated.
    nuclei_channel = 'DAPI',
    membrane_channel_list = ["CD45", "betaCatenin"], #default is None #default is None; if provide more than one channel, then they will be combined
    input_format ='Multichannel', # Phenocycler or codex
    resize_factor=1, # default is 1; if the image is too large, lower the value. Lower values will speed up the segmentation but may reduce the accuracy.
    size_cutoff = 0,
    custom_model=True) 

## Viusalizing the segmentation result

Not every dataset works equally well with all segmentation models due to differences in tissue type, structure or image quality. Therefore, it is of major importance to check the segmentation results before continuing with the data analysis. the 'show_masks' function selects random tiles of a user defined size from the image to provide examples to evaluate the segmentation quality. If the segmentation quality is not acceptable a different model should be tried. For especially challenging datasets users can also try to retrain a model specifically for their images. 

In [ ]:
overlay_data1, rgb_images1 = sp.pl.show_masks(
    seg_output=seg_output1,
    nucleus_channel = 'DAPI', # channel used for nuclei segmentation (displayed in blue)
    membrane_channel_list = ["CD45", "betaCatenin"], # additional channels to display (displayed in green - channels will be combined into one image)
    show_subsample = True, # show a random subsample of the image
    n=4, #need to be at least 2
    tilesize = 300,# number of subsamples and tilesize
    rand_seed = 42)

In [ ]:
overlay_data2, rgb_images2 = sp.pl.show_masks(
    seg_output=seg_output2,
    nucleus_channel = 'DAPI', # channel used for nuclei segmentation (displayed in blue)
    membrane_channel_list = ["CD45", "betaCatenin"], # additional channels to display (displayed in green - channels will be combined into one image)
    show_subsample = True, # show a random subsample of the image
    n=2, #need to be at least 2
    tilesize = 300, # number of subsamples and tilesize
    rand_seed = 3) 

## Save the segmentation result

After successful segmentation, the images and masks can be stored in a pickle file for later easy access (used by notebook 09, TissUUmaps). For notebooks 03.1/03.2, prefer the `{output_fname}_mask.tif` files written above by `save_mask=True` -- load them by path with `sp.hf.hf_load_masks`/`sp.hf.hf_load_image_dict` rather than unpickling this whole `seg_output` dict.

In [ ]:
#Save segmentation output
with open(output_dir + 'seg_output_tonsil1.pickle', 'wb') as f:
    pickle.dump(seg_output1, f)

with open(output_dir + 'seg_output_tonsil2.pickle', 'wb') as f:
    pickle.dump(seg_output2, f)
    
#Save the overlay of the data
with open(output_dir + 'overlay_tonsil1.pickle', 'wb') as f:
    pickle.dump(overlay_data1, f)

with open(output_dir + 'overlay_tonsil2.pickle', 'wb') as f:
    pickle.dump(overlay_data2, f)